# מנקד זעיר: אימון ב-Colab

מודל ניקוד קטן (BiLSTM על תווים, כ-5M פרמטרים, ~6MB ב-int8) שלומד מהמנקד הגדול ומהמילון ("המורה"), עם אותם טוקנים ואותם ראשי פלט כמו `nakdan.onnx` של החבילה, כך שהשירות טוען אותו בלי שינוי קוד (`tts/nakdan/nakdan.onnx`).

לפני ההרצה: להעלות ל-Google Drive, לתיקייה `otzaria-light/nakdan-tiny/dataset/`, את `part-NN.npz` ו-`meta.json` (מ-`make_dataset.py`).

Runtime ← Change runtime type ← GPU (T4 מספיק).

In [ ]:
# 1. Drive and settings
from google.colab import drive
drive.mount('/content/drive')
import os, glob, json
ROOT = '/content/drive/MyDrive/otzaria-light/nakdan-tiny'
RUN = 'v1'
OUT = f'{ROOT}/runs/{RUN}'
os.makedirs(OUT, exist_ok=True)
meta = json.load(open(f'{ROOT}/dataset/meta.json'))
parts = sorted(glob.glob(f'{ROOT}/dataset/part-*.npz'))
assert parts, 'upload the dataset parts first'
print(meta['counts']['lines'], 'examples in', len(parts), 'parts; vocab', meta['vocab_size'])
!nvidia-smi -L

In [ ]:
# 2. The data: examples (ids + labels), 2% held out by hash, padded batches sorted by length
import numpy as np, torch, random, hashlib
from torch.utils.data import Dataset, DataLoader
PAD = meta['pad']
examples = []
for p in parts:
    z = np.load(p)
    off, src = z['offsets'], z['src'].tobytes().decode('ascii')
    for i in range(len(off) - 1):
        a, b = off[i], off[i + 1]
        examples.append((src[i], z['ids'][a:b].astype(np.int64), z['nikud'][a:b].astype(np.int64),
                         z['shin'][a:b].astype(np.int64), z['stress'][a:b].astype(np.float32), z['shva'][a:b].astype(np.float32)))
random.Random(7).shuffle(examples)
dev = [e for e in examples if int(hashlib.md5(e[1].tobytes()).hexdigest()[:4], 16) % 50 == 0]
train = [e for e in examples if int(hashlib.md5(e[1].tobytes()).hexdigest()[:4], 16) % 50 != 0]
print(len(train), 'train', len(dev), 'dev')

def collate(batch):
    n = max(len(e[1]) for e in batch)
    def pad(key, fill, dtype):
        out = torch.full((len(batch), n), fill, dtype=dtype)
        for i, e in enumerate(batch): out[i, :len(e[key])] = torch.as_tensor(e[key])
        return out
    return {'src': [e[0] for e in batch], 'ids': pad(1, PAD, torch.long), 'mask': pad(1, PAD, torch.long).ne(PAD).long(),
            'nikud': pad(2, -1, torch.long), 'shin': pad(3, -1, torch.long), 'stress': pad(4, -1, torch.float), 'shva': pad(5, -1, torch.float)}

class Buckets(Dataset):
    # batches of similar length, shuffled between epochs
    def __init__(self, data, batch_size):
        order = sorted(range(len(data)), key=lambda i: len(data[i][1]))
        self.batches = [[data[j] for j in order[i:i + batch_size]] for i in range(0, len(order), batch_size)]
    def __len__(self): return len(self.batches)
    def __getitem__(self, i): return collate(self.batches[i])

BATCH = 64
train_set, dev_set = Buckets(train, BATCH), Buckets(dev, BATCH)

In [ ]:
# 3. The model: character embedding -> 3-layer BiLSTM -> the three heads of nakdan.onnx
import torch.nn as nn
class TinyNakdan(nn.Module):
    def __init__(self, vocab, emb=192, hidden=256, layers=3, dropout=0.2):
        super().__init__()
        self.emb = nn.Embedding(vocab, emb, padding_idx=PAD)
        self.drop = nn.Dropout(dropout)
        self.lstm = nn.LSTM(emb, hidden, num_layers=layers, bidirectional=True, batch_first=True, dropout=dropout)
        self.nikud = nn.Linear(2 * hidden, 29)
        self.shin = nn.Linear(2 * hidden, 2)
        self.additional = nn.Linear(2 * hidden, 3)
    def forward(self, input_ids, attention_mask=None, token_type_ids=None):
        h, _ = self.lstm(self.drop(self.emb(input_ids)))
        h = self.drop(h)
        return self.nikud(h), self.shin(h), self.additional(h)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = TinyNakdan(meta['vocab_size']).to(device)
print(sum(p.numel() for p in model.parameters()) / 1e6, 'M parameters')

In [ ]:
# 4. Training (resumes from the newest checkpoint in Drive). One epoch over 12M characters takes a few minutes on a T4.
import time, math
EPOCHS = 12
LR = 2e-3
opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
steps_total = EPOCHS * len(train_set)
sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=LR, total_steps=steps_total, pct_start=0.05)
ce = nn.CrossEntropyLoss(ignore_index=-1)
bce = nn.BCEWithLogitsLoss(reduction='none')
start_epoch = 0
ckpts = sorted(glob.glob(f'{OUT}/epoch-*.pt'))
if ckpts:
    state = torch.load(ckpts[-1], map_location=device)
    model.load_state_dict(state['model']); opt.load_state_dict(state['opt']); sched.load_state_dict(state['sched'])
    start_epoch = state['epoch'] + 1
    print('resumed from', ckpts[-1])

def evaluate():
    model.eval()
    right = {'nikud': 0, 'shin': 0, 'stress': 0, 'shva': 0}; total = dict(right)
    by_src = {}
    with torch.no_grad():
        for i in range(len(dev_set)):
            b = dev_set[i]
            n, s, a = model(b['ids'].to(device))
            pn, ps = n.argmax(-1).cpu(), s.argmax(-1).cpu()
            pst, psh = (a[..., 0] > 0).cpu().float(), (a[..., 1] > 0).cpu().float()
            m = b['nikud'] >= 0
            right['nikud'] += (pn[m] == b['nikud'][m]).sum().item(); total['nikud'] += m.sum().item()
            ms = b['shin'] >= 0
            right['shin'] += (ps[ms] == b['shin'][ms]).sum().item(); total['shin'] += ms.sum().item()
            right['stress'] += (pst[m] == b['stress'][m]).sum().item(); total['stress'] += m.sum().item()
            right['shva'] += (psh[m] == b['shva'][m]).sum().item(); total['shva'] += m.sum().item()
            for row, src in enumerate(b['src']):
                mm = m[row]
                r, t = by_src.get(src, (0, 0))
                by_src[src] = (r + (pn[row][mm] == b['nikud'][row][mm]).sum().item(), t + mm.sum().item())
    model.train()
    acc = {k: 100.0 * right[k] / max(1, total[k]) for k in right}
    acc['by_src'] = {k: round(100.0 * r / max(1, t), 2) for k, (r, t) in by_src.items()}
    return acc

model.train()
for epoch in range(start_epoch, EPOCHS):
    order = list(range(len(train_set))); random.shuffle(order)
    t0, loss_sum = time.time(), 0.0
    for k, i in enumerate(order):
        b = train_set[i]
        ids = b['ids'].to(device)
        n, s, a = model(ids)
        m = (b['nikud'] >= 0).to(device).float()
        loss = ce(n.reshape(-1, 29), b['nikud'].to(device).reshape(-1)) + 0.5 * ce(s.reshape(-1, 2), b['shin'].to(device).reshape(-1))
        loss = loss + (bce(a[..., 0], b['stress'].to(device).clamp(min=0)) * m).sum() / m.sum().clamp(min=1)
        loss = loss + (bce(a[..., 1], b['shva'].to(device).clamp(min=0)) * m).sum() / m.sum().clamp(min=1)
        opt.zero_grad(); loss.backward(); nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step(); sched.step()
        loss_sum += loss.item()
        if (k + 1) % 500 == 0: print(f'  epoch {epoch} step {k + 1}/{len(order)} loss {loss_sum / 500:.3f}'); loss_sum = 0.0
    acc = evaluate()
    print(f'epoch {epoch} done in {(time.time() - t0) / 60:.1f} min; dev letters {acc["nikud"]:.2f}% shin {acc["shin"]:.2f}% stress {acc["stress"]:.2f}% shva {acc["shva"]:.2f}% by source {acc["by_src"]}')
    torch.save({'model': model.state_dict(), 'opt': opt.state_dict(), 'sched': sched.state_dict(), 'epoch': epoch, 'acc': acc}, f'{OUT}/epoch-{epoch:02d}.pt')
    for old in sorted(glob.glob(f'{OUT}/epoch-*.pt'))[:-2]: os.remove(old)

In [ ]:
# 5. Export to ONNX with the big model's input and output names, then int8
!pip install -q onnx onnxruntime onnxscript
model.eval().cpu()
n = 40
dummy = (torch.randint(4, meta['vocab_size'], (1, n)), torch.ones(1, n, dtype=torch.long), torch.zeros(1, n, dtype=torch.long))
torch.onnx.export(model, dummy, f'{OUT}/nakdan-fp32.onnx', input_names=['input_ids', 'attention_mask', 'token_type_ids'],
                  output_names=['nikud_logits', 'shin_logits', 'additional_logits'],
                  dynamic_axes={k: {0: 'batch', 1: 'seq'} for k in ['input_ids', 'attention_mask', 'token_type_ids', 'nikud_logits', 'shin_logits', 'additional_logits']},
                  opset_version=17, dynamo=False)
from onnxruntime.quantization import quantize_dynamic, QuantType
quantize_dynamic(f'{OUT}/nakdan-fp32.onnx', f'{OUT}/nakdan.onnx', weight_type=QuantType.QInt8)
print('fp32', os.path.getsize(f'{OUT}/nakdan-fp32.onnx') / 1e6, 'MB; int8', os.path.getsize(f'{OUT}/nakdan.onnx') / 1e6, 'MB')
model.to(device)

In [ ]:
# 6. Check the int8 model with ONNX Runtime against the dev set (letters), as the service will run it
import onnxruntime as ort
sess = ort.InferenceSession(f'{OUT}/nakdan.onnx')
right = total = 0
for e in dev[:400]:
    ids = e[1][None].astype(np.int64)
    out = sess.run(None, {'input_ids': ids, 'attention_mask': np.ones_like(ids), 'token_type_ids': np.zeros_like(ids)})
    pred = out[0][0].argmax(-1)
    m = e[2] >= 0
    right += (pred[m] == e[2][m]).sum(); total += m.sum()
print(f'int8 ONNX: {100.0 * right / total:.2f}% letters on {total} letters; outputs', [o.name for o in sess.get_outputs()], [o.shape for o in sess.get_outputs()])
print('download', f'{OUT}/nakdan.onnx', 'and put it in %APPDATA%\\otzaria\\tts\\nakdan-tiny\\ with nakdan_tokenizer.json and nikud_lexicon.bin, then run eval_nikud.py with OTZARIA_NAKDAN_DIR')